# Parlance Interpreter Training — QLoRA Fine-Tuning
Fine-tunes Qwen 2.5 3B on interpreter training data (grammar, DELE/DELF, CCHI/NBCMI, legal).

**Setup:** Runtime > Change runtime type > **T4 GPU**

**Upload:** Use the Files panel (left sidebar) to upload `train.jsonl` and `val.jsonl` from `training/data/`

In [ ]:
!pip install -q transformers datasets peft bitsandbytes accelerate trl

In [ ]:
import json
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = "/content/drive/MyDrive/parlance-interpreter-model"

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import json
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR = "./parlance-interpreter-model"

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Load datasets
def load_jsonl(path):
    examples = []
    with open(path) as f:
        for line in f:
            if line.strip():
                examples.append(json.loads(line))
    return Dataset.from_list(examples)

train_ds = load_jsonl("train.jsonl")
val_ds = load_jsonl("val.jsonl")
print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

In [ ]:
# Load model with 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print("Model loaded!")

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

In [ ]:
# Format for Qwen chat template
def formatting_func(example):
    messages = example["messages"]
    text = ""
    for msg in messages:
        role = msg["role"]
        content = msg["content"]
        if role == "system":
            text += f"<|im_start|>system\n{content}<|im_end|>\n"
        elif role == "user":
            text += f"<|im_start|>user\n{content}<|im_end|>\n"
        elif role == "assistant":
            text += f"<|im_start|>assistant\n{content}<|im_end|>\n"
    return text

# Training config
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    bf16=True,
    max_seq_length=1024,
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    formatting_func=formatting_func,
)

print("Starting training...")
trainer.train()

In [ ]:
# Save the fine-tuned model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

In [ ]:
# Test the model
from peft import PeftModel

test_sentence = 'Analyze this Spanish sentence at B1 level: "Ayer yo iba al supermercado y compré muchas cosas."'

messages = [
    {"role": "system", "content": "You are a Spanish grammar coach for interpreter training. Analyze the learner's sentence at CEFR level B1. Respond with a JSON object containing: status, grammar_rule, explanation, correction, next_level_alt, target_level_alt, and tip. All example sentences must be in Spanish. Always include a register tip."},
    {"role": "user", "content": test_sentence},
]

text = ""
for msg in messages:
    text += f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n"
text += "<|im_start|>assistant\n"

inputs = tokenizer(text, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.7, do_sample=True)

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(response)

In [ ]:
# Download the model — zip it up first
!zip -r parlance-interpreter-model.zip parlance-interpreter-model/

from google.colab import files
files.download('parlance-interpreter-model.zip')